In [1]:
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

#set working directory variable to the project folder
project_folder = "/Users/md_halim_mondol/Desktop/STEPS-LLM/VLM/DSPy_project"
os.chdir(project_folder)

In [2]:
import dspy
lm = dspy.LM("openai/gpt-4o-mini")
dspy.configure(lm=lm, track_usage=False)

In [3]:
import pdfplumber

def extract_text_from_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

In [5]:
class ExtractTable(dspy.Signature):
    report_text = dspy.InputField(
        desc="Full text of a scientific report. The report may contain multiple tables."
    )
    table = dspy.OutputField(
        desc=(
            "Extract only the main raw data table from the report, not any summary table. "
            "Return the table exactly as plain text with the original column names, row order, "
            "numerical values, and units. Do not summarize, do not compute statistics, "
            "and do not output markdown."
        )
    )

In [6]:
class TableExtractor(dspy.Module):
    def __init__(self):
        super().__init__()
        self.extract = dspy.ChainOfThought(ExtractTable)

    def forward(self, report_text):
        return self.extract(report_text=report_text)

In [7]:
pdf_text = extract_text_from_pdf(f"{project_folder}/data/paper10.pdf")

extractor = TableExtractor()
result = extractor(report_text=pdf_text)

print(result.table)

Run (min) (mg/L) (mg/L) (mg/g) (mg/g) (mg/g) (L/mg) (1/(mg·min)) (-) (g) (L)
1 0.0000 41.9768 22.8689 3.6608 0.0000 4.6915 0.2123 0.0842 0.1009 0.2665 0.0510
1 2.0000 41.9768 22.8689 3.6608 1.6054 4.6915 0.2123 0.0842 0.1009 0.2665 0.0510
1 5.0000 41.9768 22.8689 3.6608 2.4820 4.6915 0.2123 0.0842 0.1009 0.2665 0.0510
1 10.0000 41.9768 22.8689 3.6608 2.9965 4.6915 0.2123 0.0842 0.1009 0.2665 0.0510
1 20.0000 41.9768 22.8689 3.6608 3.3179 4.6915 0.2123 0.0842 0.1009 0.2665 0.0510
1 30.0000 41.9768 22.8689 3.6608 3.3483 4.6915 0.2123 0.0842 0.1009 0.2665 0.0510
1 45.0000 41.9768 22.8689 3.6608 3.6608 4.6915 0.2123 0.0842 0.1009 0.2665 0.0510
1 60.0000 41.9768 22.8689 3.6608 3.6608 4.6915 0.2123 0.0842 0.1009 0.2665 0.0510
1 90.0000 41.9768 22.8689 3.6608 3.6608 4.6915 0.2123 0.0842 0.1009 0.2665 0.0510
1 120.0000 41.9768 22.8689 3.6608 3.6160 4.6915 0.2123 0.0842 0.1009 0.2665 0.0510
2 0.0000 47.2354 2.4930 2.0804 0.0334 9.7585 0.1631 0.1036 0.1149 0.5624 0.0261
2 2.0000 47.2354 2.4930 2

In [8]:
# import re

# def normalize_table_text(text: str) -> str:
#     lines = []
#     for line in text.strip().splitlines():
#         line = " ".join(line.strip().split())
#         if line:
#             lines.append(line)
#     return "\n".join(lines)

# def split_lines(text: str):
#     return [line.strip() for line in normalize_table_text(text).splitlines() if line.strip()]

# def get_header_line(text: str):
#     lines = split_lines(text)
#     return lines[0] if lines else ""

# def get_data_lines(text: str):
#     lines = split_lines(text)
#     return lines[1:] if len(lines) > 1 else []

# def extract_numeric_tokens(text: str):
#     # captures integers/floats like 1, 2.0, 0.1057
#     return re.findall(r"-?\d+(?:\.\d+)?", text)

# def safe_ratio(a, b):
#     return a / b if b else 0.0

# def table_fractional_metric(example, pred, trace=None):
#     gold_text = normalize_table_text(example.table)
#     pred_text = normalize_table_text(pred.table)

#     gold_header = get_header_line(gold_text)
#     pred_header = get_header_line(pred_text)

#     gold_rows = get_data_lines(gold_text)
#     pred_rows = get_data_lines(pred_text)

#     gold_nums = extract_numeric_tokens(gold_text)
#     pred_nums = extract_numeric_tokens(pred_text)

#     # 1) header exact match
#     header_score = 1.0 if gold_header == pred_header else 0.0

#     # 2) row-count closeness
#     if len(gold_rows) == 0:
#         row_count_score = 0.0
#     else:
#         row_count_score = max(
#             0.0,
#             1.0 - abs(len(gold_rows) - len(pred_rows)) / len(gold_rows)
#         )

#     # 3) numeric overlap in order-sensitive way
#     matched = 0
#     for g, p in zip(gold_nums, pred_nums):
#         if g == p:
#             matched += 1
#     numeric_score = safe_ratio(matched, len(gold_nums))

#     # 4) full exact match bonus
#     exact_score = 1.0 if gold_text == pred_text else 0.0

#     # weighted final score
#     score = (
#         0.30 * header_score +
#         0.20 * row_count_score +
#         0.40 * numeric_score +
#         0.10 * exact_score
#     )

#     # For bootstrapping, be stricter
#     if trace is not None:
#         return score >= 0.85

#     return score

In [9]:
# metric version 2 with more forgiving row count and numeric preservation, and a new table-like output check
import re

def normalize_table_text(text: str) -> str:
    lines = []
    for line in text.strip().splitlines():
        line = " ".join(line.strip().split())
        if line:
            lines.append(line)
    return "\n".join(lines)

def split_lines(text: str):
    return [line for line in normalize_table_text(text).splitlines() if line.strip()]

def numeric_tokens(text: str):
    return re.findall(r"-?\d+(?:\.\d+)?", text)

def token_overlap_ratio(seq1, seq2):
    if not seq1:
        return 0.0
    matched = sum(1 for a, b in zip(seq1, seq2) if a == b)
    return matched / len(seq1)

def table_fractional_metric(example, pred, trace=None):
    gold_text = normalize_table_text(example.table)
    pred_text = normalize_table_text(pred.table)

    gold_lines = split_lines(gold_text)
    pred_lines = split_lines(pred_text)

    gold_header = gold_lines[0] if gold_lines else ""
    pred_header = pred_lines[0] if pred_lines else ""

    gold_rows = gold_lines[1:] if len(gold_lines) > 1 else []
    pred_rows = pred_lines[1:] if len(pred_lines) > 1 else []

    gold_nums = numeric_tokens(gold_text)
    pred_nums = numeric_tokens(pred_text)

    # 1) header preservation: generic, no schema assumptions
    header_score = 1.0 if gold_header == pred_header else 0.0

    # 2) row-count closeness
    if len(gold_rows) == 0:
        row_score = 0.0
    else:
        row_score = max(0.0, 1.0 - abs(len(gold_rows) - len(pred_rows)) / len(gold_rows))

    # 3) numeric preservation in sequence order
    numeric_score = token_overlap_ratio(gold_nums, pred_nums)

    # 4) output is non-empty and table-like
    table_like_score = 1.0 if len(pred_lines) >= 2 and len(pred_nums) > 0 else 0.0

    # 5) exact match bonus
    exact_score = 1.0 if gold_text == pred_text else 0.0

    score = (
        0.15 * table_like_score +
        0.20 * header_score +
        0.20 * row_score +
        0.35 * numeric_score +
        0.10 * exact_score
    )

    # stricter acceptance during bootstrapping
    if trace is not None:
        return score >= 0.75

    return score

In [10]:
import os
from pathlib import Path
import dspy

data_dir = Path("data")
gold_dir = Path("gold_tables")

pdf_files = sorted(data_dir.glob("*.pdf"))

def load_text(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

trainset = []

for pdf_path in pdf_files:
    stem = pdf_path.stem                      # paper01
    gold_path = gold_dir / f"{stem}_table2.txt"

    report_text = extract_text_from_pdf(str(pdf_path))
    gold_table = load_text(gold_path)

    ex = dspy.Example( report_text=report_text, table=gold_table ).with_inputs("report_text")

    trainset.append(ex)

print("Examples:", len(trainset))

Examples: 10


In [11]:
baseline_extractor = TableExtractor()

for i, ex in enumerate(trainset, start=1):
    pred = baseline_extractor(report_text=ex.report_text)

    print(f"\n===== Example {i} =====")
    print("PREDICTED TABLE:\n")
    print(pred.table[:2000])   # prints first 2000 characters
    print("\n" + "="*50)


===== Example 1 =====
PREDICTED TABLE:

Run C0 (mg/L) Ce (mg/L) qe (mg/g) qmax (mg/g) KL (L/mg) RL (-) m (g) V (L)
1 39.8954 22.0685 3.1407 4.8165 0.0901 0.2177 0.3877 0.0683
2 9.8935 0.6564 1.4434 6.7171 0.3877 0.2068 0.4219 0.0659
3 74.8312 1.1232 4.9964 9.9384 0.4541 0.0286 0.9795 0.0664
4 58.3138 26.2386 3.9550 4.0816 0.4939 0.0336 0.6696 0.0826
5 8.4356 0.3123 0.5448 9.9890 0.3467 0.2548 0.8420 0.0565
6 68.4488 62.0225 0.8761 0.8761 0.0459 0.2416 0.2843 0.0517
7 34.6285 2.6436 2.2195 5.7595 0.2902 0.0905 0.9940 0.0690
8 76.0722 29.5974 3.6035 3.9779 0.2255 0.0551 0.8812 0.0683
9 32.7998 12.1380 5.0289 7.2909 0.1530 0.1661 0.3926 0.0956
10 75.9477 71.3903 1.1497 1.3978 0.1112 0.1059 0.3847 0.0971


===== Example 2 =====
PREDICTED TABLE:

C0 Ce Qe Qm b RL m V
Run (mg/L) (mg/L) (mg/g) (mg/g) (L/mg) (-) (g) (L)
1 10.7377 2.4289 0.4220 3.4280 0.0671 0.5811 0.8287 0.0421
2 16.6323 0.0000 1.2528 7.5260 0.4253 0.1239 0.4813 0.0362
3 21.5015 5.9987 2.2191 7.1596 0.0669 0.4101 0.4841 0.069

In [12]:
baseline_extractor = TableExtractor()

baseline_scores = []

for i, ex in enumerate(trainset, start=1):
    pred = baseline_extractor(report_text=ex.report_text)
    score = table_fractional_metric(ex, pred)
    baseline_scores.append(score)
    print(f"Example {i}: score = {score:.4f}")

baseline_avg = sum(baseline_scores) / len(baseline_scores)
print("\nBaseline fractional average:", round(baseline_avg, 4))

Example 1: score = 0.5928
Example 2: score = 0.6800
Example 3: score = 0.6800
Example 4: score = 0.4742
Example 5: score = 0.5808
Example 6: score = 0.6768
Example 7: score = 0.6204
Example 8: score = 0.3972
Example 9: score = 0.4038
Example 10: score = 0.3375

Baseline fractional average: 0.5443


In [13]:
from dspy.teleprompt import BootstrapFewShot

# keep the same split for now
train_examples = trainset[:8]
dev_examples = trainset[8:]

optimizer = BootstrapFewShot(
    metric=table_fractional_metric,
    max_bootstrapped_demos=4,
    max_labeled_demos=4,
    max_rounds=1,
)

optimized_extractor = optimizer.compile( TableExtractor(), trainset=train_examples )

print("Compilation done.")

 88%|████████▊ | 7/8 [00:00<00:00, 54.98it/s]

Bootstrapped 4 full traces after 7 examples for up to 1 rounds, amounting to 7 attempts.
Compilation done.


In [14]:
# --- BASELINE on dev ---
baseline_extractor = TableExtractor()

baseline_scores = []
for i, ex in enumerate(dev_examples, start=1):
    pred = baseline_extractor(report_text=ex.report_text)
    score = table_fractional_metric(ex, pred)
    baseline_scores.append(score)
    print(f"[BASELINE] Dev Example {i}: score = {score:.4f}")

baseline_avg = sum(baseline_scores) / len(baseline_scores)
print("\nBaseline dev fractional average:", round(baseline_avg, 4))


# --- OPTIMIZED on dev ---
optimized_scores = []
for i, ex in enumerate(dev_examples, start=1):
    pred = optimized_extractor(report_text=ex.report_text)
    score = table_fractional_metric(ex, pred)
    optimized_scores.append(score)
    print(f"[OPTIMIZED] Dev Example {i}: score = {score:.4f}")

optimized_avg = sum(optimized_scores) / len(optimized_scores)
print("\nOptimized dev fractional average:", round(optimized_avg, 4))

[BASELINE] Dev Example 1: score = 0.4038
[BASELINE] Dev Example 2: score = 0.3375

Baseline dev fractional average: 0.3706
[OPTIMIZED] Dev Example 1: score = 0.6038
[OPTIMIZED] Dev Example 2: score = 0.5631

Optimized dev fractional average: 0.5835


In [ ]:
# for i, ex in enumerate(dev_examples, start=1):
#     baseline_pred = TableExtractor()(report_text=ex.report_text)
#     optimized_pred = optimized_extractor(report_text=ex.report_text)

#     baseline_score = table_fractional_metric(ex, baseline_pred)
#     optimized_score = table_fractional_metric(ex, optimized_pred)

#     print(f"\nDEV EXAMPLE {i}")
#     print(f"Baseline score : {baseline_score:.4f}")
#     print(f"Optimized score: {optimized_score:.4f}")

#     print("\n--- BASELINE (first 1000 chars) ---")
#     print(baseline_pred.table[:1000])

#     print("\n--- OPTIMIZED (first 1000 chars) ---")
#     print(optimized_pred.table[:1000])


DEV EXAMPLE 1
Baseline score : 0.4038
Optimized score: 0.6038

--- BASELINE (first 1000 chars) ---
Run t (min) C0 (mg/L) Ceq (mg/L) qe (mg/g) qt (mg/g) qmax (mg/g) KL (L/mg) k2 (1/(mg·min)) RL (-) m (g) V (L)
1 0.0000 8.3192 1.0639 0.4466 0.0000 5.0761 0.0760 0.1010 0.6126 0.8232 0.0507
1 2.0000 8.3192 1.0639 0.4466 0.0471 5.0761 0.0760 0.1010 0.6126 0.8232 0.0507
1 5.0000 8.3192 1.0639 0.4466 0.0919 5.0761 0.0760 0.1010 0.6126 0.8232 0.0507
1 10.0000 8.3192 1.0639 0.4466 0.1388 5.0761 0.0760 0.1010 0.6126 0.8232 0.0507
1 20.0000 8.3192 1.0639 0.4466 0.1907 5.0761 0.0760 0.1010 0.6126 0.8232 0.0507
1 30.0000 8.3192 1.0639 0.4466 0.2596 5.0761 0.0760 0.1010 0.6126 0.8232 0.0507
1 45.0000 8.3192 1.0639 0.4466 0.2890 5.0761 0.0760 0.1010 0.6126 0.8232 0.0507
1 60.0000 8.3192 1.0639 0.4466 0.3348 5.0761 0.0760 0.1010 0.6126 0.8232 0.0507
1 90.0000 8.3192 1.0639 0.4466 0.3978 5.0761 0.0760 0.1010 0.6126 0.8232 0.0507
1 120.0000 8.3192 1.0639 0.4466 0.3775 5.0761 0.0760 0.1010 0.6126 0.8232

In [17]:
from IPython.display import display, HTML

def show_side_by_side(gold, baseline, optimized, title=""):
    html = f"""
    <div style="display:flex; gap:20px;">
        <div style="width:33%;">
            <h3>Gold</h3>
            <pre style="white-space: pre-wrap; border:1px solid #ccc; padding:10px;">{gold}</pre>
        </div>
        <div style="width:33%;">
            <h3>Baseline</h3>
            <pre style="white-space: pre-wrap; border:1px solid #ccc; padding:10px;">{baseline}</pre>
        </div>
        <div style="width:33%;">
            <h3>Optimized</h3>
            <pre style="white-space: pre-wrap; border:1px solid #ccc; padding:10px;">{optimized}</pre>
        </div>
    </div>
    """
    display(HTML(html))

In [ ]:
# for i, ex in enumerate(dev_examples, start=1):
#     baseline_pred = TableExtractor()(report_text=ex.report_text)
#     optimized_pred = optimized_extractor(report_text=ex.report_text)

#     print(f"\n=== DEV EXAMPLE {i} ===")

#     show_side_by_side( ex.table, baseline_pred.table, optimized_pred.table)


=== DEV EXAMPLE 1 ===



=== DEV EXAMPLE 2 ===


In [ ]:
# import dspy

# # use one dev example
# ex = dev_examples[0]

# print("\n\n===== OPTIMIZED RUN =====")
# _ = optimized_extractor(report_text=ex.report_text)
# dspy.inspect_history(n=1)



===== OPTIMIZED RUN =====




[2026-04-07T12:31:04.144988]

System message:

Your input fields are:
1. `report_text` (str): Full text of a scientific report. The report may contain multiple tables.
Your output fields are:
1. `reasoning` (str): 
2. `table` (str): Extract only the main raw data table from the report, not any summary table. Return the table exactly as plain text with the original column names, row order, numerical values, and units. Do not summarize, do not compute statistics, and do not output markdown.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## report_text ## ]]
{report_text}

[[ ## reasoning ## ]]
{reasoning}

[[ ## table ## ]]
{table}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `report_text`, produce the fields `table`.


User message:

[[ ## report_text ## ]]
Langmuir-Type Adsorption Reveals Finite Phosphate Capacity and Favorable
Retention in Saline 

In [ ]:
# generate optimization curve to see how metric score evolves with more bootstrapped demos and rounds
# Think about other metrics to imrove the optimization process for cases when no ground truth is available, or for better bootstrapping signal